In [ ]:
from pathlib import Path
import cv2
import numpy as np
from tqdm.auto import tqdm

INPUT_DIR = Path("/caa/Homes01/alaczko/Diploma/Diploma/data/rocopot_rim_256/Rims_256")               # ide tedd a sziluetteket
OUTLINE_DIR = INPUT_DIR / "outlines"     # körvonalak
BROKEN_DIR = INPUT_DIR / "broken"        # szakadozott körvonalak

LINE_THICKNESS = 2       # körvonal vastagsága (px)
N_GAPS = (4, 8)           # szakadások száma képenként: [min, max)
GAP_RADIUS = (5, 30)      # egy szakadás "sugara" px-ben: [min, max)
SEED = 42                 # None = minden futásnál más

EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
OUTLINE_DIR.mkdir(parents=True, exist_ok=True)
BROKEN_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)

In [ ]:
def make_outline(img):
    _, mask = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    outline = np.full_like(img, 255)
    cv2.drawContours(outline, contours, -1, color=0, thickness=LINE_THICKNESS)
    return outline

images = sorted(p for p in INPUT_DIR.iterdir() if p.suffix.lower() in EXTENSIONS)

print(f"{len(images)} sziluett képet találtam a {INPUT_DIR} könyvtárban.")

for path in tqdm(images, desc="Körvonalak"):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Kihagyva (nem olvasható): {path.name}")
        continue
    cv2.imwrite(str(OUTLINE_DIR / f"{path.stem}.png"), make_outline(img))

In [ ]:
def break_outline(outline):
    broken = outline.copy()
    ys, xs = np.nonzero(outline < 128)          # a vonal pixelei
    if len(xs) == 0:
        return broken
    for i in rng.choice(len(xs), rng.integers(*N_GAPS), replace=False):
        r = int(rng.integers(*GAP_RADIUS))
        cv2.circle(broken, (int(xs[i]), int(ys[i])), r, color=255, thickness=-1)
    return broken

outlines = sorted(OUTLINE_DIR.glob("*.png"))

for path in tqdm(outlines, desc="Szakadások"):
    outline = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    cv2.imwrite(str(BROKEN_DIR / path.name), break_outline(outline))

In [ ]:
NOISY_DIR = INPUT_DIR / "noisy"          # szakadozott + zajos körvonalak
NOISY_DIR.mkdir(parents=True, exist_ok=True)

N_SPECKLES = (30, 60)      # apró zajpöttyök száma képenként: [min, max)
SPECKLE_RADIUS = (1, 3)    # egy pötty sugara px-ben: [min, max)
NEAR_LINE_PROB = 0.7       # ennyi eséllyel a vonal közelébe kerül egy pötty (különben bárhova)
NEAR_LINE_MAX_DIST = 20    # "közelség" px-ben, ha a vonal mellé kerül

N_DASHES = (0, 4)          # rövid vonalkák (pl. szkennelési csíkok) száma képenként: [min, max)
DASH_LENGTH = (6, 18)      # egy vonalka hossza px-ben: [min, max)
DASH_OFFSET = (5, 25)      # a vonalka milyen távol induljon a körvonaltól

def add_noise(outline):
    noisy = outline.copy()
    h, w = noisy.shape
    ys, xs = np.nonzero(outline < 128)   # a (törött) vonal pixelei, ehhez képest helyezzük el a zajt
    has_line = len(xs) > 0

    for _ in range(rng.integers(*N_SPECKLES)):
        if has_line and rng.random() < NEAR_LINE_PROB:
            i = rng.integers(len(xs))
            cx = int(xs[i]) + int(rng.integers(-NEAR_LINE_MAX_DIST, NEAR_LINE_MAX_DIST + 1))
            cy = int(ys[i]) + int(rng.integers(-NEAR_LINE_MAX_DIST, NEAR_LINE_MAX_DIST + 1))
        else:
            cx, cy = int(rng.integers(0, w)), int(rng.integers(0, h))
        cx, cy = int(np.clip(cx, 0, w - 1)), int(np.clip(cy, 0, h - 1))
        r = int(rng.integers(*SPECKLE_RADIUS))
        cv2.circle(noisy, (cx, cy), r, color=0, thickness=-1)

    for _ in range(rng.integers(*N_DASHES)):
        if not has_line:
            break
        i = rng.integers(len(xs))
        angle = rng.uniform(0, 2 * np.pi)
        offset = rng.integers(*DASH_OFFSET)
        ox = int(xs[i] + np.cos(angle) * offset)
        oy = int(ys[i] + np.sin(angle) * offset)
        length = int(rng.integers(*DASH_LENGTH))
        dx, dy = np.cos(angle) * length / 2, np.sin(angle) * length / 2
        p1 = (int(np.clip(ox - dx, 0, w - 1)), int(np.clip(oy - dy, 0, h - 1)))
        p2 = (int(np.clip(ox + dx, 0, w - 1)), int(np.clip(oy + dy, 0, h - 1)))
        cv2.line(noisy, p1, p2, color=0, thickness=1)

    return noisy

broken_files = sorted(BROKEN_DIR.glob("*.png"))

for path in tqdm(broken_files, desc="Zajosítás"):
    broken = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    cv2.imwrite(str(NOISY_DIR / path.name), add_noise(broken))

In [ ]:
import matplotlib.pyplot as plt

# ---- Realisztikusabb zaj: pöttyök a vonal mellett + helyi megvastagodás (főleg görbületeknél) ----

TEST_NAME = None           # melyik képen teszteljünk (pl. "abc.png"); None = az első a BROKEN_DIR-ből
TEST_SEED = None           # fix seed a teszthez (pl. 0), hogy ugyanazt kapd újrafuttatáskor; None = mindig más

# Megvastagodás
N_THICK = (4, 10)          # megvastagodott szakaszok száma képenként: [min, max)
THICK_LENGTH = (8, 30)     # egy szakasz hossza a vonal mentén (px): [min, max)
THICK_EXTRA = (1.0, 3.5)   # ennyivel vastagabb a szakasz közepe (sugár px-ben): [min, max)
THICK_ONE_SIDED = 0.6      # ennyi eséllyel csak az egyik oldalra "hízik" a vonal
CURV_WINDOW = 6            # görbület számításának ablaka (pontszám a kontúr mentén)
CURV_POWER = 2.0           # mennyire preferálja a görbületeket (0 = egyenletes, nagyobb = erősebben)
END_CAP_ANGLE = 2.4        # efölötti irányváltás (radián) a szakadások végét jelzi (a kontúr ott visszafordul), azt kihagyjuk

# Kis tüskék / dudorok közvetlenül a vonalon
N_BURRS = (10, 30)         # száma képenként: [min, max)
BURR_RADIUS = (1, 3)       # sugár px-ben: [min, max)

# Pöttyök
N_DOTS = (30, 80)          # pöttyök száma képenként: [min, max)
DOT_DIST_SCALE = 3.0       # a vonaltól mért távolság exponenciális eloszlású, ennyi az átlaga (px)
DOT_RADIUS_PROBS = (0.6, 0.35, 0.05)  # 0 / 1 / 2 px sugarú pötty esélye (0 = egyetlen pixel)
FAR_DOT_PROB = 0.05        # ennyi eséllyel kerül egy pötty bárhova a képen

# Koordinátavonalak (a valós rajzokon: vízszintes a perem tetején, függőleges tengely a jobb oldalon)
TOP_LINE_PROB = 0.9        # ennyi eséllyel kerül vízszintes vonal a sziluett tetejére
RIGHT_LINE_PROB = 0.7      # ennyi eséllyel kerül függőleges vonal a jobb oldalra (csak ha van felső vonal)
AXIS_MARGIN = (2, 25)      # a függőleges vonal ennyivel a sziluett jobb széle mellett, de legfeljebb a kép szélén: [min, max)
RIGHT_LINE_LENGTH = (0.3, 1.0)   # a függőleges vonal hossza a felső vonal alatti magasság arányában: [min, max)
COORD_JITTER = 1           # a vonalvégek ennyi px-t elcsúszhatnak (enyhe ferdeség)
COORD_GAPS = (0, 3)        # szakadások száma vonalanként: [min, max)
COORD_GAP_RADIUS = (2, 6)  # egy szakadás sugara px-ben: [min, max)
SIDE_LINE_PROB = 0.3       # ennyi eséllyel kerül egy egyenes vonal a bal oldalra vagy alulra (fele-fele)
SIDE_LINE_MARGIN = (2, 25) # a vonal ennyivel a sziluett bal / alsó széle mellett, de legfeljebb a kép szélén: [min, max)
SIDE_LINE_LENGTH = (0.3, 1.0)  # a vonal hossza a képméret arányában: [min, max)

# Scribble (sraffozás, cikcakk, firka)
SCRIBBLE_PROB = 0.2        # ennyi eséllyel kerül scribble egy képre
SCRIBBLE_SIZE = (30, 90)   # a scribble kiterjedése px-ben: [min, max)
SCRIBBLE_THICKNESS = (2, 3)  # vonalvastagság px-ben: [min, max)
SCRIBBLE_EDGE_DIST = (0, 15) # a scribble széle ennyi px-re legyen a kép (véletlen) szélétől: [min, max)
SCRIBBLE_BOTTOM_LEFT_PROB = 0.6  # ennyi eséllyel a bal alsó sarokba kerül (különben egy véletlen szél mellé)


def line_contours(img):
    mask = (img < 128).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)
    return [c[:, 0, :].astype(np.float32) for c in contours if len(c) > 2 * CURV_WINDOW + 1]


def contour_curvature(pts):
    # irányváltás szöge a pont előtti és utáni CURV_WINDOW lépés között (0 = egyenes, pi = visszafordul)
    k = CURV_WINDOW
    v1 = pts - np.roll(pts, k, axis=0)
    v2 = np.roll(pts, -k, axis=0) - pts
    cos = (v1 * v2).sum(1) / (np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + 1e-6)
    return np.arccos(np.clip(cos, -1, 1))


def thicken(img, contours):
    if not contours:
        return
    # minden kontúrpont kap egy súlyt a görbülete alapján, ebből húzzuk a szakaszok középpontját
    pool = [(ci, i) for ci, c in enumerate(contours) for i in range(len(c))]
    weights = []
    for c in contours:
        curv = contour_curvature(c)
        w = curv ** CURV_POWER + 0.02
        cap = curv > END_CAP_ANGLE
        for s in range(-2 * CURV_WINDOW, 2 * CURV_WINDOW + 1):   # a szakadásvég környékét is kihagyjuk
            w[np.roll(cap, s)] = 0
        weights.append(w)
    weights = np.concatenate(weights)
    if weights.sum() == 0:
        return
    weights /= weights.sum()

    for idx in rng.choice(len(pool), int(rng.integers(*N_THICK)), p=weights):
        ci, center = pool[idx]
        pts = contours[ci]
        n = len(pts)
        half = int(rng.integers(*THICK_LENGTH)) // 2
        extra = rng.uniform(*THICK_EXTRA)
        one_sided = rng.random() < THICK_ONE_SIDED
        side = rng.choice([-1, 1])
        for j in range(-half, half + 1):
            i = (center + j) % n
            t = 1 - abs(j) / (half + 1)                         # 1 a közepén, 0 a széleken
            r = extra * np.sin(t * np.pi / 2) ** 0.8            # sima, orsó alakú profil
            r *= rng.uniform(0.8, 1.2)                          # kis egyenetlenség
            p = pts[i]
            if one_sided:
                tangent = pts[(i + 1) % n] - pts[(i - 1) % n]
                normal = np.array([-tangent[1], tangent[0]]) / (np.linalg.norm(tangent) + 1e-6)
                p = p + side * normal * r * 0.6
            cv2.circle(img, (int(round(p[0])), int(round(p[1]))), int(round(r)), color=0, thickness=-1)


def add_burrs(img, xs, ys):
    for _ in range(int(rng.integers(*N_BURRS))):
        i = rng.integers(len(xs))
        cx = int(xs[i] + rng.integers(-1, 2))
        cy = int(ys[i] + rng.integers(-1, 2))
        cv2.circle(img, (cx, cy), int(rng.integers(*BURR_RADIUS)), color=0, thickness=-1)


def add_dots(img, xs, ys):
    h, w = img.shape
    for _ in range(int(rng.integers(*N_DOTS))):
        if rng.random() < FAR_DOT_PROB:
            cx, cy = int(rng.integers(0, w)), int(rng.integers(0, h))
        else:
            i = rng.integers(len(xs))
            angle = rng.uniform(0, 2 * np.pi)
            dist = 1 + rng.exponential(DOT_DIST_SCALE)
            cx = int(round(xs[i] + np.cos(angle) * dist))
            cy = int(round(ys[i] + np.sin(angle) * dist))
        if 0 <= cx < w and 0 <= cy < h:
            r = int(rng.choice(len(DOT_RADIUS_PROBS), p=DOT_RADIUS_PROBS))
            cv2.circle(img, (cx, cy), r, color=0, thickness=-1)


def jitter(pt, w, h):
    x, y = pt
    return (int(np.clip(x + rng.integers(-COORD_JITTER, COORD_JITTER + 1), 0, w - 1)),
            int(np.clip(y + rng.integers(-COORD_JITTER, COORD_JITTER + 1), 0, h - 1)))


def draw_broken_line(img, p1, p2):
    """Egyenes vonal a körvonallal azonos vastagsággal, néhány kis szakadással."""
    h, w = img.shape
    layer = np.full_like(img, 255)
    cv2.line(layer, jitter(p1, w, h), jitter(p2, w, h), color=0, thickness=LINE_THICKNESS)
    for _ in range(int(rng.integers(*COORD_GAPS))):
        t = rng.uniform(0.1, 0.9)
        c = (int(p1[0] + t * (p2[0] - p1[0])), int(p1[1] + t * (p2[1] - p1[1])))
        cv2.circle(layer, c, int(rng.integers(*COORD_GAP_RADIUS)), color=255, thickness=-1)
    img[layer < 128] = 0


def add_coord_lines(img, clean):
    """Vízszintes vonal a sziluett legfelső pontjától jobbra, és függőleges tengely a jobb oldalon."""
    h, w = img.shape
    ys, xs = np.nonzero(clean < 128)
    if len(xs) == 0 or rng.random() >= TOP_LINE_PROB:
        return
    y_top = int(ys.min())
    x_start = int(xs[ys == y_top].min())
    x_axis = int(min(xs.max() + rng.integers(*AXIS_MARGIN), w - 1 - rng.integers(0, 3)))
    draw_broken_line(img, (x_start, y_top), (x_axis, y_top))
    if rng.random() < RIGHT_LINE_PROB:
        y_end = int(y_top + rng.uniform(*RIGHT_LINE_LENGTH) * (h - 1 - y_top))
        draw_broken_line(img, (x_axis, y_top), (x_axis, y_end))


def add_side_line(img, clean):
    """Egyenes vonal a sziluett bal oldalán (függőleges) vagy alatta (vízszintes)."""
    h, w = img.shape
    ys, xs = np.nonzero(clean < 128)
    if len(xs) == 0 or rng.random() >= SIDE_LINE_PROB:
        return
    if rng.random() < 0.5:                                        # bal oldal
        x = int(max(xs.min() - rng.integers(*SIDE_LINE_MARGIN), rng.integers(0, 3)))
        length = int(rng.uniform(*SIDE_LINE_LENGTH) * (h - 1))
        y0 = int(rng.integers(0, h - length))
        draw_broken_line(img, (x, y0), (x, y0 + length))
    else:                                                         # alul
        y = int(min(ys.max() + rng.integers(*SIDE_LINE_MARGIN), h - 1 - rng.integers(0, 3)))
        length = int(rng.uniform(*SIDE_LINE_LENGTH) * (w - 1))
        x0 = int(rng.integers(0, w - length))
        draw_broken_line(img, (x0, y), (x0 + length, y))


def add_scribble(img):
    """Kézi firka: párhuzamos sraffozás, cikcakk vagy szabad görbe a kép egyik széle mellett."""
    h, w = img.shape
    size = int(rng.integers(*SCRIBBLE_SIZE))
    if rng.random() < SCRIBBLE_BOTTOM_LEFT_PROB:                  # bal alsó sarok: a bal és az alsó szélhez is közel
        cx = size // 2 + int(rng.integers(*SCRIBBLE_EDGE_DIST))
        cy = h - 1 - size // 2 - int(rng.integers(*SCRIBBLE_EDGE_DIST))
    else:
        edge = int(rng.integers(4))                                   # 0: bal, 1: jobb, 2: felső, 3: alsó
        across = size // 2 + int(rng.integers(*SCRIBBLE_EDGE_DIST))   # a középpont távolsága a széltől
        along = int(rng.integers(size // 2, (w if edge >= 2 else h) - size // 2))
        cx, cy = [(across, along), (w - 1 - across, along), (along, across), (along, h - 1 - across)][edge]
    thickness = int(rng.integers(*SCRIBBLE_THICKNESS))
    angle = rng.uniform(0, np.pi)
    d = np.array([np.cos(angle), np.sin(angle)])          # a vonások iránya
    n = np.array([-d[1], d[0]])                           # erre haladunk vonásról vonásra
    kind = rng.choice(["hatch", "zigzag", "free"])
    if kind == "hatch":
        k = int(rng.integers(5, 15))
        for t in np.linspace(-size / 2, size / 2, k):
            half = size / 2 * rng.uniform(0.4, 1.0)
            off = rng.uniform(-size / 6, size / 6)
            c = np.array([cx, cy]) + n * t + d * off
            cv2.line(img, tuple(int(v) for v in c - d * half), tuple(int(v) for v in c + d * half), 0, thickness)
    else:
        k = int(rng.integers(5, 14))
        if kind == "zigzag":
            pts = [np.array([cx, cy]) + n * t + d * (size / 2 * (1 if j % 2 else -1) * rng.uniform(0.6, 1.0))
                   for j, t in enumerate(np.linspace(-size / 2, size / 2, k))]
        else:
            pts = [np.array([cx, cy], float)]
            for _ in range(k * 3):
                pts.append(np.clip(pts[-1] + rng.normal(0, size / 6, 2), [cx - size / 2, cy - size / 2], [cx + size / 2, cy + size / 2]))
        pts = np.clip(np.array(pts), 0, [w - 1, h - 1]).astype(np.int32)
        cv2.polylines(img, [pts], isClosed=False, color=0, thickness=thickness, lineType=cv2.LINE_AA)


def add_noise_v2(outline, clean=None):
    """outline: szakadozott körvonal; clean: az ép körvonal (a koordinátavonalak helyéhez), ha nincs, az outline-t használja."""
    noisy = outline.copy()
    if not (outline < 128).any():
        return noisy
    add_coord_lines(noisy, outline if clean is None else clean)   # előbb a vonalak, hogy ugyanazt a zajt kapják
    add_side_line(noisy, outline if clean is None else clean)
    ys, xs = np.nonzero(noisy < 128)
    thicken(noisy, line_contours(noisy))
    add_burrs(noisy, xs, ys)
    if rng.random() < SCRIBBLE_PROB:
        add_scribble(noisy)
    add_dots(noisy, xs, ys)
    return noisy


# ---- Teszt egy képen ----
if TEST_SEED is not None:
    rng = np.random.default_rng(TEST_SEED)

test_path = BROKEN_DIR / TEST_NAME if TEST_NAME else sorted(BROKEN_DIR.glob("*.png"))[0]
broken = cv2.imread(str(test_path), cv2.IMREAD_GRAYSCALE)
noisy = add_noise_v2(broken, cv2.imread(str(OUTLINE_DIR / test_path.name), cv2.IMREAD_GRAYSCALE))

diff = cv2.cvtColor(broken, cv2.COLOR_GRAY2RGB)
diff[(noisy < 128) & (broken >= 128)] = (220, 0, 0)    # pirossal az újonnan hozzáadott pixelek

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (im, title) in zip(axes, [(broken, "Szakadozott (bemenet)"), (noisy, "Zajos"), (diff, "Hozzáadott zaj (piros)")]):
    ax.imshow(im, cmap="gray" if im.ndim == 2 else None, interpolation="nearest")
    ax.set_title(title); ax.axis("off")
plt.suptitle(test_path.name)
plt.tight_layout()
plt.show()

In [ ]:
NOISY_DIR = INPUT_DIR / "noisy"
NOISY_DIR.mkdir(parents=True, exist_ok=True)
broken_files = sorted(BROKEN_DIR.glob("*.png"))

for path in tqdm(broken_files, desc="Zajosítás"):
    broken = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    clean = cv2.imread(str(OUTLINE_DIR / path.name), cv2.IMREAD_GRAYSCALE)
    cv2.imwrite(str(NOISY_DIR / path.name), add_noise_v2(broken, clean))

In [ ]:
from pathlib import Path
import random
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

TRAIN_INPUT_DIR = Path("/caa/Homes01/alaczko/Diploma/Diploma/data/rocopot_rim_256/Rims_256/noisy")      # bemenet: törött körvonalak
SILHOUETTE_DIR = Path("/caa/Homes01/alaczko/Diploma/Diploma/data/rocopot_rim_256/Rims_256")         # ground truth: teli sziluett (ezt tanulja a modell)
OUTLINE_DIR = Path("/caa/Homes01/alaczko/Diploma/Diploma/data/rocopot_rim_256/Rims_256/outlines")   # hibátlan körvonalak (csak összehasonlításhoz)
IMG_SIZE = 256            # 16-tal osztható legyen
BATCH_SIZE = 64
EPOCHS = 100
LR = 1e-3
VAL_RATIO = 0.2
MODEL_PATH = "unet_silhouette.pt"

torch.backends.cudnn.benchmark = True

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Eszköz:", device)
random.seed(42); torch.manual_seed(42)

In [ ]:
def load(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return (img < 128).astype(np.float32)     # vonal / sziluett belseje = 1, háttér = 0

def mask_to_outline(mask, thickness=2, min_area=50):
    """Teli maszkból (bool) körvonal (bool), ugyanúgy, ahogy a ground truth körvonalak készültek; az apró foltokat eldobja."""
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    contours = [c for c in contours if cv2.contourArea(c) >= min_area]
    outline = np.zeros(mask.shape, np.uint8)
    cv2.drawContours(outline, contours, -1, color=1, thickness=thickness)
    return outline.astype(bool)

In [ ]:


class OutlineDataset(Dataset):
    def __init__(self, files, augment=False):
        self.augment = augment
        self.files = files
        self.X = np.stack([load(f) for f in tqdm(files, desc="Betöltés")])
        self.Y = np.stack([load(SILHOUETTE_DIR / f.name) for f in files])
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        x, y = self.X[i], self.Y[i]
        if self.augment:
            k = random.randint(0, 3)
            x, y = np.rot90(x, k), np.rot90(y, k)
            if random.random() < 0.5:
                x, y = np.fliplr(x), np.fliplr(y)
        return torch.from_numpy(x.copy())[None], torch.from_numpy(y.copy())[None]

files = sorted(p for p in TRAIN_INPUT_DIR.glob("*.png") if (SILHOUETTE_DIR / p.name).exists())
random.shuffle(files)
n_val = max(1, int(len(files) * VAL_RATIO))
train_ds = OutlineDataset(files[n_val:], augment=True)
val_ds = OutlineDataset(files[:n_val])
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE)
print(f"Train: {len(train_ds)}, val: {len(val_ds)}")

def show_training_samples(ds, n=3):
    """Ellenőrzés tanítás előtt: n véletlen bemenet + sziluett pár a datasetből, és hogy a bemenet miben tér el a sziluett körvonalától."""
    import matplotlib.pyplot as plt
    print(f"Bemenet: {TRAIN_INPUT_DIR}\nGround truth: {SILHOUETTE_DIR}")
    idx = random.sample(range(len(ds)), n)
    fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n), squeeze=False)
    for row, i in zip(axes, idx):
        x, sil = ds.X[i] > 0.5, ds.Y[i] > 0.5
        y = mask_to_outline(sil)
        diff = np.ones((*x.shape, 3))
        diff[x & y] = (0.6, 0.6, 0.6)        # szürke: mindkettőben vonal
        diff[x & ~y] = (0.85, 0, 0)          # piros: csak a bemenetben (zaj, vastagodás)
        diff[~x & y] = (0, 0.6, 0)           # zöld: csak a GT-ben (szakadás)
        for ax, (im, title) in zip(row, [(1 - x, f"Bemenet: {ds.files[i].name}"),
                                         (1 - sil, "Ground truth (sziluett)"),
                                         (diff, f"Eltérés: {(x & ~y).sum()} zaj px, {(~x & y).sum()} szakadás px")]):
            ax.imshow(im, cmap="gray" if im.ndim == 2 else None, interpolation="nearest")
            ax.set_title(title, fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()

def show_epoch_sample(model, ds, epoch, i=0, handle=None):
    """Tanítás közben: ugyanarra a validációs képre a modell kimenete, a küszöbölt sziluett + ground truth.
    Az első híváskor új ábrát nyit, utána ugyanazt frissíti (a visszaadott handle-t kell visszaadni neki)."""
    import matplotlib.pyplot as plt
    from IPython.display import display
    x, y = ds[i]
    model.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model(x[None].to(device)).float())[0, 0].cpu().numpy()
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, (im, title) in zip(axes, [(1 - x[0].numpy(), f"Bemenet: {ds.files[i].name}"),
                                      (1 - prob, f"Kimenet (epoch {epoch})"),
                                      (1 - (prob > 0.5), "Küszöbölt sziluett"),
                                      (1 - y[0].numpy(), "Ground truth (sziluett)")]):
        ax.imshow(im, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(title, fontsize=9); ax.axis("off")
    plt.tight_layout()
    plt.close(fig)                      # ne rajzolja ki magától, mi tesszük be a handle-be
    if handle is None:
        return display(fig, display_id=True)
    handle.update(fig)
    return handle


In [ ]:
def block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
    )

class UNet(nn.Module):
    def __init__(self, ch=(32, 64, 128, 256)):
        super().__init__()
        self.downs = nn.ModuleList()
        cin = 1
        for c in ch:
            self.downs.append(block(cin, c)); cin = c
        self.bottleneck = block(ch[-1], ch[-1] * 2)
        self.ups = nn.ModuleList()
        self.up_blocks = nn.ModuleList()
        cin = ch[-1] * 2
        for c in reversed(ch):
            self.ups.append(nn.ConvTranspose2d(cin, c, 2, stride=2))
            self.up_blocks.append(block(c * 2, c)); cin = c
        self.out = nn.Conv2d(ch[0], 1, 1)

    def forward(self, x):
        skips = []
        for d in self.downs:
            x = d(x); skips.append(x); x = F.max_pool2d(x, 2)
        x = self.bottleneck(x)
        for up, blk, s in zip(self.ups, self.up_blocks, reversed(skips)):
            x = blk(torch.cat([up(x), s], dim=1))
        return self.out(x)

def dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits)
    inter = (p * target).sum(dim=(1, 2, 3))
    union = p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return 1 - ((2 * inter + eps) / (union + eps)).mean()

def loss_fn(logits, target):
    return F.binary_cross_entropy_with_logits(logits, target) + dice_loss(logits, target)

In [ ]:
model = UNet().to(device)

In [ ]:
show_training_samples(train_ds)

model = UNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=5, min_lr=1e-6)
scaler = torch.amp.GradScaler(enabled=(device == "cuda"))
best_val = float("inf")

ES_PATIENCE = 15   # ES: ennyi javulás nélküli epoch után leáll
bad_epochs = 0     # ES

plot_handle = None                                   # az epochonkénti ábra, ugyanazt frissítjük
epoch_bar = tqdm(range(1, EPOCHS + 1), desc="Tanítás")

for epoch in epoch_bar:
    model.train(); train_loss = 0
    for x, y in tqdm(train_dl, desc=f"Epoch {epoch}", leave=False):
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device, enabled=(device == "cuda")):
            loss = loss_fn(model(x), y)
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        train_loss += loss.item() * len(x)
    train_loss /= len(train_ds)

    model.eval(); val_loss = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device, enabled=(device == "cuda")):
                val_loss += loss_fn(model(x), y).item() * len(x)
    val_loss /= len(val_ds)

    lr_before = opt.param_groups[0]["lr"]     # adaptive LR: elmentjük lépés előtt
    sched.step(val_loss)
    lr_after = opt.param_groups[0]["lr"]

    if val_loss < best_val:
        best_val = val_loss
        bad_epochs = 0                                          # ES: javult, számláló nullázása
        torch.save(model.state_dict(), MODEL_PATH)
    else:
        bad_epochs += 1                                         # ES

    # egy sorban frissülő állapot a progress bar végén
    epoch_bar.set_postfix(train=f"{train_loss:.4f}", val=f"{val_loss:.4f}", best=f"{best_val:.4f}",
                          lr=f"{lr_after:.1e}", ES=f"{bad_epochs}/{ES_PATIENCE}")
    if lr_after < lr_before:
        epoch_bar.write(f"Epoch {epoch}: lr csökkentve {lr_before:.2e} -> {lr_after:.2e}")
    plot_handle = show_epoch_sample(model, val_ds, epoch, handle=plot_handle)

    if bad_epochs >= ES_PATIENCE:                               # ES
        epoch_bar.write(f"Early stopping: {ES_PATIENCE} epoch óta nincs javulás.")
        break


In [ ]:
import matplotlib.pyplot as plt
import random

TEST_DIR = Path("/caa/Homes01/alaczko/Diploma/Diploma/data/pipeline/segmentation_run/repaired")   # ide a javítandó képek
FIXED_DIR = Path("/caa/Homes01/alaczko/Diploma/Diploma/data/pipeline/segmentation_run/fixed")     # ide kerülnek a javított képek
DEBUG_DIR = FIXED_DIR.parent / "fixed_debug"                                                  # ide a before/after összehasonlítások
FIXED_DIR.mkdir(exist_ok=True)
DEBUG_DIR.mkdir(exist_ok=True)

SAVE_DEBUG = True      # mentsen-e debug képeket
N_SHOW = 4             # ennyi véletlen példát jelenít meg a notebookban a végén
THRESHOLD = 0.5        # efölötti valószínűségnél a sziluett részének tekintjük a pixelt

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

def fix_image(path):
    orig = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    x = torch.from_numpy(load(path))[None, None].to(device)
    with torch.no_grad():
        prob = torch.sigmoid(model(x))[0, 0].cpu().numpy()
    prob = cv2.resize(prob, (orig.shape[1], orig.shape[0]), interpolation=cv2.INTER_LINEAR)
    fixed = np.where(prob > THRESHOLD, 0, 255).astype(np.uint8)   # teli sziluett: fekete alakzat fehér háttéren
    return orig, prob, fixed

def label(img, text):
    img = img.copy()
    cv2.rectangle(img, (0, 0), (img.shape[1], 22), (255, 255, 255), thickness=-1)
    cv2.putText(img, text, (5, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
    return img

def make_debug(orig, prob, fixed):
    orig_line = orig < 128
    sil = fixed < 128

    orig_bgr = cv2.cvtColor(orig, cv2.COLOR_GRAY2BGR)
    fixed_bgr = cv2.cvtColor(fixed, cv2.COLOR_GRAY2BGR)
    prob_bgr = cv2.applyColorMap((np.clip(prob, 0, 1) * 255).astype(np.uint8), cv2.COLORMAP_JET)

    diff = np.full((*orig.shape, 3), 255, dtype=np.uint8)
    diff[sil] = (150, 230, 150)                          # halványzöld: a sziluett
    diff[orig_line] = (0, 0, 0)                          # fekete: az eredeti vonalak rajta

    sep = np.full((orig.shape[0], 4, 3), 128, dtype=np.uint8)
    panels = [
        label(orig_bgr, "eredeti"),
        label(prob_bgr, "valoszinuseg"),
        label(fixed_bgr, "sziluett"),
        label(diff, f"atfedes, {sil.sum()} px"),
    ]
    row = panels[0]
    for p in panels[1:]:
        row = np.hstack([row, sep, p])
    return row, sil.sum()

paths = sorted(TEST_DIR.glob("*.jpg"))
stats = []

for path in tqdm(paths, desc="Javítás"):
    orig, prob, fixed = fix_image(path)
    cv2.imwrite(str(FIXED_DIR / path.name), fixed)
    if SAVE_DEBUG:
        debug, area = make_debug(orig, prob, fixed)
        cv2.imwrite(str(DEBUG_DIR / f"{path.stem}.png"), debug)   # png, hogy ne legyen jpg műtermék
        stats.append((path.name, area))

if stats:
    stats.sort(key=lambda s: s[1])
    print("Legkisebb sziluettek (gyanús, itt valószínűleg nem sikerült kitölteni):")
    for name, area in stats[:10]:
        print(f"  {name}: {area} px")

# Néhány véletlen példa megjelenítése
for path in random.sample(paths, min(N_SHOW, len(paths))):
    debug, _ = make_debug(*fix_image(path))
    plt.figure(figsize=(18, 5))
    plt.imshow(cv2.cvtColor(debug, cv2.COLOR_BGR2RGB))
    plt.title(path.name)
    plt.axis("off")
    plt.show()

In [ ]:
import random
n = 40
sample_files = random.sample(val_ds.files, n)
fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
for row, f in zip(axes, sample_files):
    name = f.name
    for ax, (p, title) in zip(row, [(TEST_DIR / name, "broken"),
                                     (FIXED_DIR / name, "fixed (U-Net)"),
                                     (OUTLINE_DIR / name, "Ground truth")]):
        ax.imshow(cv2.imread(str(p), cv2.IMREAD_GRAYSCALE), cmap="gray")
        ax.set_title(title)
        ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

IMG_PATH = Path("/caa/Homes01/alaczko/Diploma/Diploma/data/pipeline/run_2909___2/repaired/00_Selection ML__1976_Schedivy Zivilstadt GB_Sektoren 1-2 bis 21-22__Sektor 1-2__027.jpg")   # a javítandó kép

# Modell betöltése (ha még nincs betöltve)
model = UNet().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

# Javítás
orig = cv2.imread(str(IMG_PATH), cv2.IMREAD_GRAYSCALE)
x = torch.from_numpy(load(IMG_PATH))[None, None].to(device)
with torch.no_grad():
    prob = torch.sigmoid(model(x))[0, 0].cpu().numpy()
prob = cv2.resize(prob, (orig.shape[1], orig.shape[0]), interpolation=cv2.INTER_LINEAR)
fixed = np.where(prob > 0.5, 0, 255).astype(np.uint8)   # teli sziluett

# Plot
panels = [(orig, "Törött"), (fixed, "Sziluett")]
gt_path = SILHOUETTE_DIR / IMG_PATH.name
if gt_path.exists():
    panels.append((cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE), "Ground truth"))

fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
for ax, (im, title) in zip(axes, panels):
    ax.imshow(im, cmap="gray"); ax.set_title(title); ax.axis("off")
plt.show()